In [ ]:
import pandas as pd
import os
import glob

# CATATAN: Pastikan Anda telah menginstal library yang diperlukan.
# Buka terminal di Visual Studio Code dan jalankan perintah ini:
# pip install pandas

def combine_social_media_data(file_paths, output_filename="data_gabungan_final.csv"):
    """
    Menggabungkan beberapa file CSV dari berbagai platform media sosial (Twitter, TikTok, YouTube)
    menjadi satu DataFrame tunggal dan menyimpannya sebagai file CSV.

    Fungsi ini akan menstandardisasi kolom-kolom umum seperti username, teks,
    tanggal, dan jumlah suka.

    Args:
        file_paths (list): Daftar path ke file-file CSV yang akan digabungkan.
        output_filename (str): Nama file CSV untuk output.
    """
    all_data_frames = []

    # Definisikan pemetaan kolom berdasarkan kolom yang sebenarnya ada di data Anda
    column_mappings = {
        # Data Twitter 1: conversation_id_str, created_at, favorite_count, full_text, id_str, image_url, in_reply_to_screen_name, lang, location, quote_count, reply_count, retweet_count, tweet_url, user_id_str, username
        'datatwitter1': {
            'username': 'username',
            'full_text': 'text',
            'created_at': 'timestamp',
            'favorite_count': 'likes_count',
            'tweet_url': 'url',
            'id_str': 'id',
            'retweet_count': 'retweet_count',
            'reply_count': 'reply_count',
            'quote_count': 'quote_count',
            'lang': 'language',
            'location': 'location'
        },
        # Data Twitter 2: url, verified, username, timestamp, text, searchQuery, quotedTweet
        'datatwitter2': {
            'username': 'username',
            'text': 'text',
            'timestamp': 'timestamp',
            'url': 'url',
            'verified': 'verified',
            'searchQuery': 'search_query',
            'quotedTweet': 'quoted_tweet'
        },
        # Data YouTube: cid, replyToCid, type, publishedTimeText, comment, author, authorIsChannelOwner, replyCount, voteCount, hasCreatorHeart, videoId, pageUrl, commentsCount, title
        'datayt1': {
            'author': 'username',
            'comment': 'text',
            'publishedTimeText': 'timestamp',
            'voteCount': 'likes_count',
            'pageUrl': 'url',
            'cid': 'id',
            'replyCount': 'reply_count',
            'authorIsChannelOwner': 'is_channel_owner',
            'hasCreatorHeart': 'has_creator_heart',
            'videoId': 'video_id',
            'title': 'video_title'
        },
        # Data TikTok: videoWebUrl, submittedVideoUrl, input, cid, createTime, createTimeISO, text, diggCount, likedByAuthor, pinnedByAuthor, repliesToId, replyCommentTotal, uid, uniqueId, avatarThumbnail, mentions, detailedMentions
        'datatiktok1': {
            'uniqueId': 'username',
            'text': 'text',
            'createTimeISO': 'timestamp',
            'diggCount': 'likes_count',
            'videoWebUrl': 'url',
            'cid': 'id',
            'replyCommentTotal': 'reply_count',
            'likedByAuthor': 'liked_by_author',
            'pinnedByAuthor': 'pinned_by_author',
            'uid': 'user_id',
            'mentions': 'mentions'
        }
    }
    
    # Kolom standar yang ingin kita simpan di file final
    final_columns = [
        'id', 'source_platform', 'username', 'timestamp', 'text', 'likes_count', 'url',
        'reply_count', 'retweet_count', 'quote_count', 'language', 'location', 'verified',
        'search_query', 'quoted_tweet', 'is_channel_owner', 'has_creator_heart', 'video_id',
        'video_title', 'liked_by_author', 'pinned_by_author', 'user_id', 'mentions'
    ]

    print(f"Memproses {len(file_paths)} file...")

    for file_path in file_paths:
        # ========== LOKASI PEMBACAAN PATH FILE ==========
        # Di sini kode memeriksa apakah file benar-benar ada di path yang diberikan
        if not os.path.exists(file_path):
            print(f"File tidak ditemukan: {file_path}")
            print(f"Path lengkap yang dicari: {os.path.abspath(file_path)}")
            continue
            
        filename = os.path.basename(file_path).lower()
        source_platform = "unknown"
        
        try:
            # Tentukan platform dan mapping berdasarkan nama file yang tepat
            filename_lower = filename.replace('.csv', '').lower()
            
            if filename_lower == 'datatwitter1':
                source_platform = 'Twitter'
                mapping = column_mappings['datatwitter1']
            elif filename_lower == 'datatwitter2':
                source_platform = 'Twitter'
                mapping = column_mappings['datatwitter2']
            elif filename_lower == 'datayt1':
                source_platform = 'YouTube'
                mapping = column_mappings['datayt1']
            elif filename_lower == 'datatiktok1':
                source_platform = 'TikTok'
                mapping = column_mappings['datatiktok1']
            else:
                print(f"Peringatan: Tidak dapat menentukan platform untuk file '{filename}'. Menggunakan mapping default.")
                source_platform = filename.split('.')[0].title()
                mapping = {}

            # Baca file CSV
            print(f"Membaca file: {file_path}")
            df = pd.read_csv(file_path, encoding='utf-8')
            print(f"File berhasil dibaca. Jumlah baris: {len(df)}, Kolom: {list(df.columns)}")
            
            # Tambahkan kolom untuk menandai sumber data
            df['source_platform'] = source_platform
            
            # Ganti nama kolom berdasarkan pemetaan yang sudah didefinisikan
            if mapping:
                df.rename(columns=mapping, inplace=True)
            
            # Filter DataFrame agar hanya berisi kolom standar
            # Kolom yang tidak ada di DataFrame akan diabaikan
            existing_cols = [col for col in final_columns if col in df.columns]
            
            # Jika tidak ada kolom yang cocok, tetap ambil beberapa kolom pertama
            if not existing_cols or len(existing_cols) <= 1:  # Hanya source_platform
                print(f"Peringatan: Tidak ada kolom standar yang ditemukan di {filename}")
                print(f"Kolom yang tersedia: {list(df.columns)}")
                # Ambil semua kolom yang ada plus source_platform
                df_filtered = df
            else:
                df_filtered = df[existing_cols]
            
            all_data_frames.append(df_filtered)
            print(f"-> Berhasil memproses: {filename} sebagai data {source_platform}")

        except Exception as e:
            print(f"Error saat memproses file {file_path}: {e}")
            import traceback
            traceback.print_exc()

    if not all_data_frames:
        print("Tidak ada data yang berhasil diproses. File output tidak akan dibuat.")
        return

    # Gabungkan semua DataFrame menjadi satu
    print("\nMenggabungkan semua data...")
    combined_df = pd.concat(all_data_frames, ignore_index=True, sort=False)

    # Pastikan semua kolom final ada, isi dengan None jika tidak ada
    for col in final_columns:
        if col not in combined_df.columns:
            combined_df[col] = None
    
    # Atur ulang urutan kolom sesuai standar (hanya jika semua kolom ada)
    available_final_cols = [col for col in final_columns if col in combined_df.columns]
    if available_final_cols:
        # Tambahkan kolom lain yang mungkin ada tapi tidak ada di final_columns
        other_cols = [col for col in combined_df.columns if col not in final_columns]
        combined_df = combined_df[available_final_cols + other_cols]
    
    # Simpan DataFrame gabungan ke file CSV baru
    try:
        combined_df.to_csv(output_filename, index=False, encoding='utf-8')
        print(f"\nProses selesai! Data gabungan telah disimpan di: {output_filename}")
        print(f"Total baris data yang digabungkan: {len(combined_df)}")
        print("Ringkasan data per platform:")
        print(combined_df['source_platform'].value_counts())
        
        # Tampilkan preview data
        print("\nPreview 5 baris pertama:")
        print(combined_df.head())
        
    except Exception as e:
        print(f"Error saat menyimpan file: {e}")
        import traceback
        traceback.print_exc()

# --- CARA PENGGUNAAN ---
if __name__ == '__main__':
    # ========== LOKASI UTAMA UNTUK MENGATUR PATH FILE ==========
    # OPSI 1: File ada di direktori yang sama dengan script Python
    file_list = [
        'datatiktok1.csv', 
        'Datatwitter1.csv', 
        'datatwitter2.csv', 
        'datayt1.csv'
    ]
    
    # OPSI 2: File ada di direktori lain (contoh untuk Windows)
    # file_list = [
    #     r'C:\Users\NamaUser\Documents\datatiktok1.csv',
    #     r'C:\Users\NamaUser\Documents\Datatwitter1.csv',
    #     r'C:\Users\NamaUser\Documents\datatwitter2.csv',
    #     r'C:\Users\NamaUser\Documents\datayt1.csv'
    # ]
    
    # OPSI 3: File ada di direktori lain (contoh untuk Mac/Linux)
    # file_list = [
    #     '/Users/namauser/Documents/datatiktok1.csv',
    #     '/Users/namauser/Documents/Datatwitter1.csv',
    #     '/Users/namauser/Documents/datatwitter2.csv',
    #     '/Users/namauser/Documents/datayt1.csv'
    # ]
    
    # OPSI 4: Menggunakan path relatif dari direktori script
    # file_list = [
    #     './data/datatiktok1.csv',
    #     './data/Datatwitter1.csv', 
    #     './data/datatwitter2.csv',
    #     './data/datayt1.csv'
    # ]
    
    # OPSI 5: Mencari semua file csv di direktori tertentu
    # import glob
    # file_list = glob.glob('*.csv')  # Semua file .csv di direktori saat ini
    # file_list = glob.glob('./data/*.csv')  # Semua file .csv di folder 'data'
    
    # Cek file mana yang benar-benar ada
    existing_files = []
    for file_path in file_list:
        if os.path.exists(file_path):
            existing_files.append(file_path)
            print(f"File ditemukan: {file_path}")
        else:
            print(f"File tidak ditemukan: {file_path}")
    
    if not existing_files:
        print("Tidak ada file yang ditemukan. Pastikan file-file berada di direktori yang sama dengan script ini.")
        print("File yang dicari:")
        for f in file_list:
            print(f"  - {f}")
    else:
        # Filter file agar tidak membaca file outputnya sendiri jika skrip dijalankan lagi
        output_basename = "data_gabungan_final.csv"
        existing_files = [f for f in existing_files if os.path.basename(f) != output_basename]
        
        if existing_files:
            print(f"\nMemulai proses penggabungan {len(existing_files)} file...")
            combine_social_media_data(existing_files)
        else:
            print("Tidak ada file yang bisa diproses setelah filtering.")

File ditemukan: datatiktok1.csv
File ditemukan: datatwitter1.csv
File ditemukan: datatwitter2.csv
File ditemukan: dataYT1.csv

Memulai proses penggabungan 4 file...
Memproses 4 file...
Membaca file: datatiktok1.csv
File berhasil dibaca. Jumlah baris: 2256, Kolom: ['videoWebUrl', 'submittedVideoUrl', 'input', 'cid', 'createTime', 'createTimeISO', 'text', 'diggCount', 'likedByAuthor', 'pinnedByAuthor', 'repliesToId', 'replyCommentTotal', 'uid', 'uniqueId', 'avatarThumbnail', 'mentions', 'detailedMentions']
-> Berhasil memproses: datatiktok1.csv sebagai data TikTok
Membaca file: datatwitter1.csv
File berhasil dibaca. Jumlah baris: 1092, Kolom: ['conversation_id_str', 'created_at', 'favorite_count', 'full_text', 'id_str', 'image_url', 'in_reply_to_screen_name', 'lang', 'location', 'quote_count', 'reply_count', 'retweet_count', 'tweet_url', 'user_id_str', 'username']
-> Berhasil memproses: datatwitter1.csv sebagai data Twitter
Membaca file: datatwitter2.csv
File berhasil dibaca. Jumlah bari